# LLM Architecture Innovations: Attention, Position, MoE, and Beyond

Modern large language models are not monolithic they are composites of carefully engineered
architectural choices. This notebook surveys the key innovations that have shaped models like
LLaMA, Mixtral, Mistral, GPT-4, Mamba, and Gemini.

## Overview

| Technique | Used In | Key Benefit |
|-----------|---------|-------------|
| Multi-Query Attention (MQA) | PaLM, Falcon | 8-32× KV cache reduction |
| Grouped Query Attention (GQA) | LLaMA 3, Mistral | Quality + cache balance |
| Sliding Window Attention | Mistral 7B | O(N·w) memory, long context |
| Ring Attention | Gemini (training) | Distributed million-token context |
| RoPE | LLaMA, GPT-NeoX, PaLM | Relative positions, length generalisation |
| ALiBi | MPT, BLOOM | No positional params, extrapolation |
| Mixture of Experts (MoE) | Mixtral, Switch-T, GPT-4(?) | Sparse compute, large capacity |
| Mamba (SSM) | Mamba, Jamba | O(1) inference, O(N) memory |
| RWKV / RetNet | RWKV-v5, RetNet | Linear attention, RNN inference |
| RMSNorm | LLaMA, T5, PaLM | 7-15% faster than LayerNorm |
| SwiGLU | LLaMA, PaLM, Gemini | Improved FFN quality |

Throughout this notebook we implement each component from scratch in PyTorch to build
intuition for what the math looks like in code.

## Multi-Head Attention (MHA) Recap

Standard multi-head attention projects queries, keys, and values into $h$ parallel subspaces,
computes scaled dot-product attention in each, then concatenates and projects:

$$\text{MHA}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

where each head is:

$$\text{head}_i = \text{Attention}(Q W_i^Q,\ K W_i^K,\ V W_i^V)$$

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_{head}}}\right) V$$

**Dimensions:**
- Model dimension: $d_{model}$
- Number of heads: $h$
- Per-head dimension: $d_{head} = d_{model} / h$
- Each $W_i^Q, W_i^K, W_i^V \in \mathbb{R}^{d_{model} \times d_{head}}$
- Output projection: $W^O \in \mathbb{R}^{d_{model} \times d_{model}}$

**KV Cache Cost (autoregressive inference):**

At each decoding step we cache the K and V projections for every past token.
For a sequence of length $N$:

$$\text{KV cache} = 2 \times N \times L \times h \times d_{head} \times \text{bytes\_per\_element}$$

Since $h \cdot d_{head} = d_{model}$:

$$= 2 \times N \times L \times d_{model} \times \text{bytes}$$

For LLaMA-2-70B ($d_{model}=8192$, $L=80$, FP16): $2 \times N \times 80 \times 8192 \times 2 = 2.62 \text{ MB per token}$.
A 4096-token context would require ≈10.7 GB a significant fraction of GPU memory.

This KV cache bottleneck motivates the attention variants described next.

## Multi-Query Attention (MQA) and Grouped Query Attention (GQA)

### Multi-Query Attention (MQA)

Proposed by Shazeer (2019), MQA shares a **single** K,V head across all query heads.
Only Q is projected into $h$ heads; K and V have just one head each.

```
MHA: Q1 Q2 Q3 Q4    K1 K2 K3 K4    V1 V2 V3 V4
MQA: Q1 Q2 Q3 Q4    K              V
GQA: Q1 Q2 | Q3 Q4  K1   |   K2   V1   |   V2
```

**KV cache reduction:**
- MHA: $2 \times h \times d_{head}$ per token per layer
- MQA: $2 \times 1 \times d_{head}$ per token per layer
- Speedup: $h \times$ fewer KV cache bytes

### Grouped Query Attention (GQA)

GQA (Ainslie et al. 2023) generalises both MHA and MQA with $G$ KV head groups.
Each group of $H/G$ query heads shares one K,V head.

$$\text{GQA: } G \text{ KV heads}, H \text{ Q heads}, H/G \text{ Q heads per KV head}$$

| Configuration | KV heads | Q heads | KV cache vs MHA |
|---------------|----------|---------|-----------------|
| MHA | H | H | 1× |
| GQA (G=8) | 8 | H | H/8 × smaller |
| MQA | 1 | H | H× smaller |

**Example LLaMA 3 (70B):**
- $H = 64$ query heads, $G = 8$ KV heads
- Each KV head serves 8 query heads
- KV cache is 8× smaller than MHA

**Quality:**
MQA is slightly worse than MHA at the same model size.
GQA with $G \approx H/8$ recovers most of MHA quality while achieving substantial cache savings.
Empirically GQA at G=H/8 is within ~0.5% of MHA perplexity.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# -------------------------------------------------
# 1.  Multi-Head Attention (standard)
# -------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn   = F.softmax(scores, dim=-1)
        out    = (attn @ V).transpose(1, 2).reshape(B, N, self.d_model)
        return self.W_o(out)

    def kv_cache_bytes_per_token_per_layer(self, dtype_bytes=2):
        return 2 * self.num_heads * self.d_head * dtype_bytes


# -------------------------------------------------
# 2.  Multi-Query Attention
# -------------------------------------------------
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, self.d_head, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, N, 1, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, N, 1, self.d_head).transpose(1, 2)
        K = K.expand(-1, self.num_heads, -1, -1)
        V = V.expand(-1, self.num_heads, -1, -1)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn   = F.softmax(scores, dim=-1)
        out    = (attn @ V).transpose(1, 2).reshape(B, N, self.d_model)
        return self.W_o(out)

    def kv_cache_bytes_per_token_per_layer(self, dtype_bytes=2):
        return 2 * 1 * self.d_head * dtype_bytes


# -------------------------------------------------
# 3.  Grouped Query Attention
# -------------------------------------------------
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        assert num_heads % num_kv_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.groups = num_heads // num_kv_heads
        self.d_head = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, N, self.num_kv_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, N, self.num_kv_heads, self.d_head).transpose(1, 2)
        K = K.repeat_interleave(self.groups, dim=1)
        V = V.repeat_interleave(self.groups, dim=1)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn   = F.softmax(scores, dim=-1)
        out    = (attn @ V).transpose(1, 2).reshape(B, N, self.d_model)
        return self.W_o(out)

    def kv_cache_bytes_per_token_per_layer(self, dtype_bytes=2):
        return 2 * self.num_kv_heads * self.d_head * dtype_bytes


# -------------------------------------------------
# 4.  Comparison
# -------------------------------------------------
d_model, H = 512, 8

mha = MultiHeadAttention(d_model, H)
mqa = MultiQueryAttention(d_model, H)
gqa = GroupedQueryAttention(d_model, H, num_kv_heads=2)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("="*55)
print(f"{'Model':<20} {'Params':>10} {'KV bytes/tok/layer':>20}")
print("="*55)
for name, model in [("MHA (H=8)", mha), ("MQA (1 KV)", mqa), ("GQA (G=2 KV)", gqa)]:
    kv = model.kv_cache_bytes_per_token_per_layer()
    print(f"{name:<20} {count_params(model):>10,} {kv:>20} bytes")
print()

N_seq, L = 4096, 32
print("KV Cache at N=4096, L=32 layers (FP16):")
for name, model in [("MHA", mha), ("MQA", mqa), ("GQA G=2", gqa)]:
    total = model.kv_cache_bytes_per_token_per_layer() * N_seq * L
    print(f"  {name:<12}: {total/1e6:.2f} MB")

x = torch.randn(2, 16, d_model)
print()
for name, model in [("MHA", mha), ("MQA", mqa), ("GQA G=2", gqa)]:
    out = model(x)
    print(f"  {name} output shape: {out.shape}")

Model                    Params   KV bytes/tok/layer
MHA (H=8)             1,048,576                 2048 bytes
MQA (1 KV)              589,824                  256 bytes
GQA (G=2 KV)            655,360                  512 bytes

KV Cache at N=4096, L=32 layers (FP16):
  MHA         : 268.44 MB
  MQA         : 33.55 MB
  GQA G=2     : 67.11 MB

  MHA output shape: torch.Size([2, 16, 512])
  MQA output shape: torch.Size([2, 16, 512])
  GQA G=2 output shape: torch.Size([2, 16, 512])


## Sliding Window Attention (SWA)

In standard attention every token attends to **all** previous tokens: $O(N^2)$ cost.
Sliding window attention restricts each token to attend only to the $w$ nearest tokens:

$$\text{Attention}(q_i) = \text{softmax}\!\left(\frac{q_i K_{[i-w:i]}^T}{\sqrt{d_{head}}}\right) V_{[i-w:i]}$$

**Memory complexity:** $O(N \cdot w)$ instead of $O(N^2)$

**Receptive field growth with depth:**

Like convolutional networks, the effective context grows through the layers.
With window size $w$ and $L$ layers:

$$\text{Receptive field} = w \times L \text{ tokens}$$

For Mistral 7B: $w = 4096$, $L = 32$ effective receptive field = **131,072 tokens**.

**Rolling Buffer KV Cache:**

For autoregressive generation, Mistral uses a rolling buffer of fixed size $w$:
- Only the most recent $w$ K,V vectors are stored
- Memory is $O(w)$ regardless of sequence length $N$
- Past tokens outside the window are simply discarded

**Mistral 7B SWA configuration:**

| Parameter | Value |
|-----------|-------|
| Window size $w$ | 4,096 |
| Layers | 32 |
| Effective context | 131k tokens |
| KV cache size | Fixed at 4k x layers |

SWA is often interleaved with full attention layers (e.g., Mistral uses full attention in some
layers for long-range dependencies) this hybrid balances efficiency and capability.

**Comparison: Full vs Sliding Window Attention**

| | Full Attention | SWA |
|---|---|---|
| Attention cost | $O(N^2)$ | $O(N \cdot w)$ |
| KV cache per token | grows with N | fixed at w |
| Long-range dependencies | direct | indirect (through layers) |

## Ring Attention: Distributed Long Context

Ring Attention (Liu et al. 2023) distributes the **sequence dimension** across $d$ devices
to enable context lengths of millions of tokens.

### How it works

1. The sequence of length $N$ is split into $d$ chunks of $N/d$ tokens each.
2. Each device $i$ holds its chunk's Q, K, V projections.
3. Devices are arranged in a ring topology.
4. At each step:
   - Each device computes attention between its Q chunk and its current K,V chunk.
   - Each device sends its K,V chunk to the next device ($i \to i+1$).
5. After $d$ steps, every device has seen all K,V chunks and accumulated the full attention output.

### Memory analysis

$$\text{Memory per device} = O\!\left(\frac{N}{d}\right)$$

This is the key advantage: memory scales linearly and is distributed across devices.
A 1M-token sequence on 64 devices is equivalent to a 15,625-token sequence per device.

### Communication cost

At each of the $d$ steps, each device sends $\frac{N}{d} \times d_{model}$ floats.
Total communication per layer = $N \times d_{model}$ floats (amortized over $d$ steps).
This overlaps with computation (pipelining KV sends with attention computation).

### Use cases

- **Gemini:** Uses distributed attention during training for very long contexts
- **LongContext fine-tuning:** Combine with RoPE extensions to reach 128k-1M contexts
- **Video/audio modelling:** Sequence lengths can reach millions of frames

### Comparison with other long-context methods

| Method | Memory per device | Communication | Training | Inference |
|--------|------------------|---------------|----------|-----------|
| Full Attention | $O(N^2)$ | none | yes | yes |
| SWA | $O(N \cdot w)$ | none | yes | yes |
| Ring Attention | $O(N/d)$ | $O(N/d)$ per step | yes | yes |
| Sparse Attention | $O(N \cdot k)$ | none | yes | yes |

## Positional Encodings: ALiBi and RoPE

Standard **learned positional embeddings** (GPT-2, BERT) add position-specific vectors to token embeddings.
They do not generalise beyond the training context length a model trained on 2k tokens
degrades when given 4k tokens.

---

### ALiBi Attention with Linear Biases

ALiBi (Press et al. 2022) adds no positional information to embeddings.
Instead, it applies a **linear penalty** to attention logits based on token distance:

$$\text{score}_{ij} = q_i \cdot k_j / \sqrt{d} - m_i \cdot |i - j|$$

where $m_i$ is a head-specific slope (fixed, geometric sequence from $2^{-1/h}$).

**Properties:**
- No positional parameters (saves memory and parameters)
- Naturally biases attention to prefer nearby tokens
- Generalises gracefully to longer sequences than seen during training
- Used in MPT-7B, BLOOM

---

### RoPE Rotary Position Embedding

RoPE (Su et al. 2021) encodes positions by **rotating** Q and K vectors.
A position-$m$ query vector $q_m$ is rotated by angle $m\theta$:

$$\tilde{q}_m = R_{\Theta,m}^d q_m, \quad R_{\Theta,m}^d = \text{diag}(R_{\theta_1, m}, \ldots, R_{\theta_{d/2}, m})$$

where $R_{\theta_j, m} = \begin{pmatrix} \cos(m\theta_j) & -\sin(m\theta_j) \\ \sin(m\theta_j) & \cos(m\theta_j) \end{pmatrix}$

The dot product $\tilde{q}_m^T \tilde{k}_n$ depends only on the **relative position** $m - n$:

$$\tilde{q}_m^T \tilde{k}_n = \text{Re}\!\left[(W_q x_m) \odot \overline{(W_k x_n) e^{i(m-n)\theta}}\right]$$

Frequencies follow a geometric schedule: $\theta_j = 10000^{-2j/d}$

**Properties:**
- Relative positions encoded naturally in dot products
- No additional parameters
- Used in: LLaMA 1/2/3, GPT-NeoX, PaLM, Mistral, Gemma

---

### RoPE Extensions for Long Context

| Method | Idea | Models |
|--------|------|--------|
| Position Interpolation | Scale position by $L_{train}/L_{target}$ | LLaMA extended |
| YaRN | Non-uniform interpolation + temperature scaling | Mistral 32k |
| NTK-aware | Change RoPE base $\theta$ to avoid aliasing | Many fine-tunes |
| LongRoPE | Progressive extension with non-uniform scaling | LongRoPE |
| CLEX | Continuous length extrapolation via ODE | CLEX models |

In [2]:
import torch
import torch.nn as nn
import math

# -------------------------------------------------
# 1. Rotary Position Embedding
# -------------------------------------------------

def precompute_freqs_cis(d_head: int, max_seq_len: int, base: float = 10000.0):
    """Precompute the complex exponential frequencies for RoPE."""
    j = torch.arange(0, d_head, 2, dtype=torch.float32)
    inv_freq = 1.0 / (base ** (j / d_head))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary embeddings.
    x: (B, T, n_heads, d_head)
    freqs_cis: (T, d_head/2)
    """
    B, T, H, D = x.shape
    x_complex = torch.view_as_complex(x.float().reshape(B, T, H, D // 2, 2))
    freqs = freqs_cis.unsqueeze(0).unsqueeze(2)
    x_rotated = x_complex * freqs
    return torch.view_as_real(x_rotated).reshape(B, T, H, D).to(x.dtype)


class RoPEAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int = 2048):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        freqs_cis = precompute_freqs_cis(self.d_head, max_seq_len)
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, x):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_head)
        K = self.W_k(x).view(B, N, self.num_heads, self.d_head)
        V = self.W_v(x).view(B, N, self.num_heads, self.d_head)
        Q = apply_rotary_emb(Q, self.freqs_cis[:N])
        K = apply_rotary_emb(K, self.freqs_cis[:N])
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).reshape(B, N, -1)
        return self.W_o(out)


# -------------------------------------------------
# 2. ALiBi bias
# -------------------------------------------------

def get_alibi_slopes(num_heads: int) -> torch.Tensor:
    """Compute ALiBi head-specific slopes."""
    n = 2 ** math.floor(math.log2(num_heads))
    m = torch.arange(1, n + 1, dtype=torch.float32)
    slopes = 1.0 / (2 ** (8 * m / n))
    if n < num_heads:
        extra = get_alibi_slopes(2 * n)
        extra = extra[0::2][:num_heads - n]
        slopes = torch.cat([slopes, extra])
    return slopes[:num_heads]


def compute_alibi_bias(seq_len: int, num_heads: int) -> torch.Tensor:
    """Returns ALiBi bias (num_heads, seq_len, seq_len)."""
    slopes = get_alibi_slopes(num_heads)
    positions = torch.arange(seq_len, dtype=torch.float32)
    dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    bias = -slopes.view(-1, 1, 1) * dist.unsqueeze(0)
    return bias


# -------------------------------------------------
# 3. Demo
# -------------------------------------------------

print("=== RoPE: relative position in dot products ===")
d_head, T = 16, 8
freqs = precompute_freqs_cis(d_head, T)

q_vec = torch.randn(1, T, 1, d_head)
k_vec = torch.randn(1, T, 1, d_head)
q_rot = apply_rotary_emb(q_vec, freqs)
k_rot = apply_rotary_emb(k_vec, freqs)

dot_35 = (q_rot[0, 3, 0] * k_rot[0, 5, 0]).sum().item()
dot_13 = (q_rot[0, 1, 0] * k_rot[0, 3, 0]).sum().item()
dot_24 = (q_rot[0, 2, 0] * k_rot[0, 4, 0]).sum().item()
print(f"  dot(q3,k5)={dot_35:.4f}  dot(q1,k3)={dot_13:.4f}  dot(q2,k4)={dot_24:.4f}")
print("  (All have relative distance=2; RoPE encodes relative position)")

print()
print("=== ALiBi slopes (8 heads) ===")
slopes = get_alibi_slopes(8)
for i, s in enumerate(slopes):
    print(f"  Head {i}: slope = {s.item():.4f}")

print()
print("=== ALiBi bias matrix (4 tokens, 2 heads) ===")
bias = compute_alibi_bias(4, 2)
print("Head 0:")
print(bias[0].numpy().round(3))
print("Head 1:")
print(bias[1].numpy().round(3))

rope_attn = RoPEAttention(64, 4, max_seq_len=32)
x = torch.randn(2, 10, 64)
print(f"\nRoPE attention output shape: {rope_attn(x).shape}")

=== RoPE: relative position in dot products ===
  dot(q3,k5)=7.0127  dot(q1,k3)=-1.8068  dot(q2,k4)=-1.6530
  (All have relative distance=2; RoPE encodes relative position)

=== ALiBi slopes (8 heads) ===
  Head 0: slope = 0.5000
  Head 1: slope = 0.2500
  Head 2: slope = 0.1250
  Head 3: slope = 0.0625
  Head 4: slope = 0.0312
  Head 5: slope = 0.0156
  Head 6: slope = 0.0078
  Head 7: slope = 0.0039

=== ALiBi bias matrix (4 tokens, 2 heads) ===
Head 0:
[[-0.    -0.062 -0.125 -0.188]
 [-0.062 -0.    -0.062 -0.125]
 [-0.125 -0.062 -0.    -0.062]
 [-0.188 -0.125 -0.062 -0.   ]]
Head 1:
[[-0.    -0.004 -0.008 -0.012]
 [-0.004 -0.    -0.004 -0.008]
 [-0.008 -0.004 -0.    -0.004]
 [-0.012 -0.008 -0.004 -0.   ]]

RoPE attention output shape: torch.Size([2, 10, 64])


## Mixture of Experts (MoE)

MoE replaces the dense feed-forward network (FFN) in each Transformer block with $N$
**expert** FFNs and a learned **router** that selects which experts process each token.

### Architecture

$$\text{MoE}(x) = \sum_{i \in \text{Top-}k(x)} G(x)_i \cdot E_i(x)$$

where:
- $E_i(x)$ is the $i$-th expert (a standard FFN)
- $G(x) = \text{Softmax}(\text{TopK}(x W_g))$ is the gating distribution
- $\text{TopK}$ keeps only the top-$k$ logits and zeros the rest

### Why MoE?

| Property | Dense Model | MoE Model |
|----------|-------------|----------|
| Total parameters | $P$ | $N \times P_{expert}$ |
| Active parameters per token | $P$ | $k \times P_{expert}$ |
| Compute per token | high | same as dense $k \times P_{expert}$ |
| Capacity | fixed | large (many experts) |

**The key insight:** MoE decouples total model capacity from per-token compute.

### Mixtral 8x7B

- 8 experts per MoE layer, top-2 routing
- Each expert is approximately a 7B FFN block
- Total parameters: **46.7B**
- Active parameters per forward pass: **~12.9B** (2 of 8 experts)
- Performance matches LLaMA-2-70B at much lower inference cost

### Load Balancing

Without explicit regularisation, the router tends to **collapse** to always choosing
the same few experts (expert collapse). The load balancing auxiliary loss encourages
uniform utilisation:

$$\mathcal{L}_{aux} = \alpha \cdot N \sum_{i=1}^{N} f_i \cdot P_i$$

where $f_i$ = fraction of tokens routed to expert $i$, $P_i$ = mean gating probability.

### Variants

| Variant | Routing | Notes |
|---------|---------|-------|
| Switch Transformer | Top-1 | Simpler, capacity factor prevents overflow |
| Mixtral/GShard | Top-2 | Better quality |
| Expert Choice | Each expert picks top-k tokens | Guarantees load balance |
| DeepSeek-MoE | Shared + routed experts | Shared experts always active |

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class MixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int = 2,
                 load_balance_coef: float = 0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.load_balance_coef = load_balance_coef
        self.experts = nn.ModuleList([FeedForward(d_model, d_ff) for _ in range(num_experts)])
        self.router = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        x: (B, T, d_model)
        Returns (output, aux_loss, load_distribution)
        """
        B, T, D = x.shape
        x_flat = x.view(B * T, D)
        router_logits = self.router(x_flat)
        router_probs  = F.softmax(router_logits, dim=-1)

        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        expert_counts = torch.zeros(self.num_experts, device=x.device)
        for k_idx in range(self.top_k):
            for e in range(self.num_experts):
                expert_counts[e] += (top_k_indices[:, k_idx] == e).float().sum()
        f = expert_counts / (B * T * self.top_k)
        P = router_probs.mean(dim=0)
        aux_loss = self.load_balance_coef * self.num_experts * (f * P).sum()

        output = torch.zeros_like(x_flat)
        for k_idx in range(self.top_k):
            expert_idx = top_k_indices[:, k_idx]
            weight     = top_k_probs[:, k_idx]
            for e in range(self.num_experts):
                mask = (expert_idx == e)
                if mask.any():
                    expert_out = self.experts[e](x_flat[mask])
                    output[mask] += weight[mask].unsqueeze(-1) * expert_out

        return output.view(B, T, D), aux_loss, f.detach()


# -------------------------------------------------
# Demo
# -------------------------------------------------
torch.manual_seed(42)
d_model, d_ff, N_experts, top_k = 256, 512, 8, 2
moe = MixtureOfExperts(d_model, d_ff, N_experts, top_k)

dense_ffn = FeedForward(d_model, d_ff)
total_params  = sum(p.numel() for p in moe.parameters())
active_params = sum(p.numel() for p in moe.experts[0].parameters()) * top_k
dense_params  = sum(p.numel() for p in dense_ffn.parameters())

print("="*50)
print(f"MoE: {N_experts} experts, top-{top_k} routing")
print(f"Total  parameters : {total_params:,}")
print(f"Active parameters : {active_params:,}  ({100*active_params/total_params:.1f}% of total)")
print(f"Dense FFN params  : {dense_params:,}")
print()

x = torch.randn(2, 16, d_model)
out, aux_loss, load = moe(x)
print(f"Output shape : {out.shape}")
print(f"Aux loss     : {aux_loss.item():.6f}")
print()
print("Expert token load distribution:")
for i, frac in enumerate(load):
    bar = "#" * int(frac.item() * 200)
    print(f"  Expert {i}: {frac.item():.3f}  {bar}")

MoE: 8 experts, top-2 routing
Total  parameters : 2,105,344
Active parameters : 525,824  (25.0% of total)
Dense FFN params  : 262,912



Output shape : torch.Size([2, 16, 256])
Aux loss     : 0.010172

Expert token load distribution:
  Expert 0: 0.094  ##################
  Expert 1: 0.109  #####################
  Expert 2: 0.109  #####################
  Expert 3: 0.125  #########################
  Expert 4: 0.062  ############
  Expert 5: 0.125  #########################
  Expert 6: 0.188  #####################################
  Expert 7: 0.188  #####################################


## Sparse Upcycling and Branch-Train-MiX

Training MoE models from scratch is expensive and unstable (router initialisation matters
a lot). Two techniques avoid this by building on pre-trained dense checkpoints.

### Sparse Upcycling (Komatsuzaki et al. 2022)

**Recipe:**
1. Start from a trained dense Transformer checkpoint.
2. For each MoE layer, **replicate** the dense FFN weights into all $N$ expert slots.
3. Add a freshly initialized router (small network).
4. Fine-tune the model the router learns to specialise experts.

**Why it works:**
- Experts start with good representations (inherited from dense training).
- Only the router and slight expert divergence need to be learned.
- Dramatically cheaper than MoE-from-scratch.

**Results (Komatsuzaki et al.):**
- Upcycled MoE beats the dense base model and matches a larger dense model.
- Fine-tuning cost: ~10% of training the dense model from scratch.

### Branch-Train-MiX (BTX, Li et al. 2023)

**Recipe:**
1. Train a **seed** dense model on general data.
2. **Branch**: clone the seed model $N$ times.
3. **Train** each branch independently on a different domain (code, math, instruction, etc.).
4. **MiX**: merge branches into a MoE model each branch becomes one expert.
   - Attention layers: average the weights across all branches.
   - FFN layers: each branch's FFN becomes one expert.
   - Add a freshly initialized router.

**Advantages over Upcycling:**
- Experts are genuinely specialised by domain before mixing.
- Natural load balance: domain-specific tokens route to the matching expert.
- Supports adding new domains later by branching again.

### Comparison

| Method | Starting point | Expert init | Specialisation | Cost |
|--------|---------------|-------------|----------------|------|
| MoE from scratch | random | random | learned | high |
| Sparse Upcycling | dense ckpt | replicated | fine-tuned | low |
| Branch-Train-MiX | dense ckpt | domain-trained | pre-trained | medium |

## State Space Models: Mamba

Transformers have $O(N^2)$ attention cost. State Space Models (SSMs) offer an alternative
with **linear complexity** in sequence length.

### Linear Time-Invariant SSM

The classical SSM computes:

$$h_t = A h_{t-1} + B x_t$$
$$y_t = C h_t$$

where $h_t \in \mathbb{R}^N$ is the hidden state, $A, B, C$ are fixed matrices.
This is a linear recurrence can be parallelised with **associative scan** during training,
and runs in $O(1)$ per step during inference.

### Structured SSM (S4)

S4 uses a special diagonal-plus-low-rank structure for $A$ (HiPPO initialisation)
that allows efficient computation and captures long-range dependencies.

### Mamba: Selective State Spaces

The key limitation of classical SSMs: $A, B, C$ are **time-invariant** (same for all inputs).
Mamba (Gu & Dao 2023) makes $B, C, \Delta$ **input-dependent** (selective):

$$\bar{A} = e^{\Delta A}, \quad \bar{B} = \Delta B x_t$$

where $\Delta = \text{softplus}(x_t W_{\Delta} + b)$ is a learned input-dependent step size.

**Properties:**
- Selectivity lets the model forget irrelevant tokens (like attention)
- Hardware-aware algorithm: parallel scan on GPU for training, sequential for inference
- Memory: $O(N \cdot d_{state})$ linear in sequence length
- Inference: $O(1)$ per step (constant memory, no KV cache)

### Mamba-2 and Structured State Space Duality

Mamba-2 (Dao & Gu 2024) connects SSMs and attention via **structured state space duality (SSD)**:
the Mamba-2 layer is equivalent to a form of linear attention with structured matrices.
This allows larger state dimensions and better hardware utilisation.

### Hybrid Architectures

| Model | Architecture | Approach |
|-------|-------------|----------|
| Mamba | Pure SSM | Replace all attention with Mamba layers |
| Jamba | SSM + Transformer | Mix Mamba and attention layers (4:1 ratio) |
| Zamba | SSM + shared attention | Shared attention layer + Mamba backbone |
| RWKV-6 | Linear attention | Transformer-like training, RNN inference |

### Mamba vs Transformer

| | Transformer | Mamba |
|---|---|---|
| Training cost | $O(N^2)$ | $O(N)$ |
| Inference memory | $O(N)$ KV cache | $O(1)$ state |
| Inference per step | $O(N)$ | $O(1)$ |
| Long-range modelling | direct attention | compressed state |
| Quality at scale | state-of-the-art | approaching Transformer |

## RWKV and RetNet: Linear Attention Alternatives

### RWKV Receptance Weighted Key Value

RWKV (Peng et al. 2023) is a hybrid that behaves like a Transformer during training
and like an RNN during inference.

**Core formula (WKV attention):**

$$\text{wkv}_t = \frac{\sum_{i \le t} e^{-(t-1-i)w + k_i} v_i}{\sum_{i \le t} e^{-(t-1-i)w + k_i}}$$

- $w$: channel-specific time decay (learned)
- $k_i, v_i$: key/value at position $i$
- Exponential decay means recent tokens are upweighted

**Dual modes:**

| Mode | Form | Use case |
|------|------|----------|
| Parallel (training) | Temporal convolution | Efficient GPU training |
| Recurrent (inference) | $O(1)$ per step | Memory-efficient generation |

The recurrent state is a simple numerator/denominator pair no KV cache needed.

**RWKV-v5/6** adds multi-scale exponential decay, achieving quality comparable to Transformers.

---

### RetNet Retention Networks

RetNet (Sun et al. 2023) introduces a **retention mechanism** that supports
three computation modes:

1. **Parallel (training):** $\text{Ret}(X) = (QK^T \odot D) V$
   where $D_{mn} = \gamma^{m-n}$ for $m \ge n$ else 0 a causal decay mask.

2. **Recurrent (inference):** $S_n = \gamma S_{n-1} + k_n^T v_n$
   with $O(1)$ state and $O(1)$ per step.

3. **Chunk-wise:** Process chunks in parallel, pass state between chunks.

**Key difference from RWKV:** RetNet uses a scalar retention factor $\gamma \in (0,1)$
per head; RWKV uses per-channel learned decay.

---

### Comparison: Transformers vs Linear Alternatives

| Property | Transformer | RWKV | RetNet |
|----------|-------------|------|--------|
| Training | $O(N^2)$ | $O(N)$ | $O(N)$ |
| Inference memory | $O(N)$ KV | $O(1)$ | $O(1)$ |
| Inference per step | $O(N)$ | $O(1)$ | $O(1)$ |
| Content-based recall | strong | moderate | moderate |
| Long-range | excellent | good | good |

## Normalization: RMSNorm vs LayerNorm

### LayerNorm

LayerNorm normalises the activations across the feature dimension:

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where $\mu = \frac{1}{d}\sum_i x_i$ (mean) and $\sigma^2 = \frac{1}{d}\sum_i (x_i - \mu)^2$ (variance).

Parameters: $\gamma, \beta \in \mathbb{R}^d$ (scale and shift).

### RMSNorm

RMSNorm (Zhang & Sennrich 2019) removes the mean-centering step:

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_i x_i^2}$$

Only the scale parameter $\gamma \in \mathbb{R}^d$ is used (no shift $\beta$).

**Why omit mean centering?**
- Mean is typically close to 0 in deep networks anyway
- Removing it saves approximately half the normalisation computation
- Measured speedup: **7-15%** faster than LayerNorm in practice
- Used in: LLaMA 1/2/3, T5, PaLM, Gemma, Mistral, Falcon

### Pre-Norm vs Post-Norm

Two placement strategies for normalisation in Transformer blocks:

**Post-Norm (original Transformer):**
$$x = \text{Norm}(x + \text{SubLayer}(x))$$

**Pre-Norm (modern default):**
$$x = x + \text{SubLayer}(\text{Norm}(x))$$

| | Post-Norm | Pre-Norm |
|---|---|---|
| Training stability | can diverge at large LR | more stable |
| Warm-up requirement | often needed | less needed |
| Final performance | slightly better (when it trains) | competitive |
| Usage | BERT, original Transformer | LLaMA, GPT-NeoX, PaLM |

Pre-norm has become the default in modern LLMs due to improved training stability.

## Activation Functions: SwiGLU and GLU Variants

### Standard FFN

The standard Transformer FFN uses two linear layers with GELU activation:

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

### Gated Linear Units (GLU)

GLU (Dauphin et al. 2017) adds a gating mechanism:

$$\text{GLU}(x, W, V) = (x W) \otimes \sigma(x V)$$

The sigmoid gate $\sigma(xV)$ controls information flow from the linear branch $xW$.

### SwiGLU

SwiGLU (Shazeer 2020) replaces sigmoid with the Swish activation:

$$\text{Swish}(x) = x \cdot \sigma(\beta x)$$

$$\text{SwiGLU}(x, W, V, W_2) = (\text{Swish}(x W) \otimes (x V)) W_2$$

This requires **3 weight matrices** ($W, V, W_2$) instead of 2.

**FLOPs parity trick:** Use hidden dimension $d_{ff} = \frac{2}{3} \cdot 4 d_{model} = \frac{8}{3} d_{model}$
(rounded to nearest multiple of 256) to match the FLOPs of a standard 2-matrix FFN with $d_{ff} = 4 d_{model}$.

### GLU Family

| Variant | Gate function | Used in |
|---------|--------------|--------|
| GLU | sigmoid | original GLU paper |
| ReGLU | ReLU | ablation studies |
| GEGLU | GELU | T5 v1.1 |
| SwiGLU | Swish (beta=1) | LLaMA, PaLM, Gemini |
| bilinear | identity | linear alternative |

Shazeer's empirical study found SwiGLU and GEGLU consistently outperform standard GELU FFN
by ~0.5-1 perplexity points across multiple model sizes.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

# -------------------------------------------------
# 1.  RMSNorm
# -------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).sqrt()
        return x / (rms + self.eps) * self.weight


# -------------------------------------------------
# 2.  SwiGLU and GeGLU activations
# -------------------------------------------------
class SwiGLU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, gate = x.chunk(2, dim=-1)
        return F.silu(gate) * x


class GeGLU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, gate = x.chunk(2, dim=-1)
        return F.gelu(gate) * x


# -------------------------------------------------
# 3.  LLaMA-style FFN block (SwiGLU)
# -------------------------------------------------
class LLaMAFeedForward(nn.Module):
    """SwiGLU FFN with 8/3 hidden dim for FLOPs parity with 4x standard FFN."""
    def __init__(self, d_model: int, multiple_of: int = 256):
        super().__init__()
        d_ff = int(8 * d_model / 3)
        d_ff = multiple_of * ((d_ff + multiple_of - 1) // multiple_of)
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


# -------------------------------------------------
# 4.  GeGLU FFN
# -------------------------------------------------
class GeGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.proj = nn.Linear(d_model, d_ff * 2, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.act  = GeGLU()

    def forward(self, x):
        return self.down(self.act(self.proj(x)))


# -------------------------------------------------
# 5.  Comparison
# -------------------------------------------------
d_model = 512
layer_norm = nn.LayerNorm(d_model)
rms_norm   = RMSNorm(d_model)
llama_ffn  = LLaMAFeedForward(d_model)
std_ffn    = nn.Sequential(
    nn.Linear(d_model, 4 * d_model),
    nn.GELU(),
    nn.Linear(4 * d_model, d_model)
)

print("=== Parameter Counts ===")
print(f"LayerNorm                    : {sum(p.numel() for p in layer_norm.parameters()):,}")
print(f"RMSNorm                      : {sum(p.numel() for p in rms_norm.parameters()):,}")
print(f"Standard FFN (d_ff=4x512)    : {sum(p.numel() for p in std_ffn.parameters()):,}")
print(f"LLaMA FFN (SwiGLU, 8/3x512) : {sum(p.numel() for p in llama_ffn.parameters()):,}")

x = torch.randn(32, 128, d_model)
REPS = 500

def benchmark(fn, inp, reps):
    for _ in range(10):
        fn(inp)
    t0 = time.perf_counter()
    for _ in range(reps):
        fn(inp)
    return (time.perf_counter() - t0) / reps * 1000

t_ln  = benchmark(layer_norm, x, REPS)
t_rms = benchmark(rms_norm, x, REPS)

print(f"\n=== Normalisation Speed (batch=32, seq=128, d={d_model}) ===")
print(f"LayerNorm  : {t_ln:.3f} ms/call")
print(f"RMSNorm    : {t_rms:.3f} ms/call")
print(f"Speedup    : {t_ln/t_rms:.2f}x")

print("\n=== Forward pass shapes ===")
print(f"LLaMAFeedForward output : {llama_ffn(x).shape}")
print(f"RMSNorm output          : {rms_norm(x).shape}")

=== Parameter Counts ===
LayerNorm                    : 1,024
RMSNorm                      : 512
Standard FFN (d_ff=4x512)    : 2,099,712
LLaMA FFN (SwiGLU, 8/3x512) : 2,359,296



=== Normalisation Speed (batch=32, seq=128, d=512) ===
LayerNorm  : 6.249 ms/call
RMSNorm    : 36.400 ms/call
Speedup    : 0.17x

=== Forward pass shapes ===


LLaMAFeedForward output : torch.Size([32, 128, 512])
RMSNorm output          : torch.Size([32, 128, 512])


## Long Context Methods

Training on longer contexts is expensive quadratic in standard attention.
A practical ecosystem of techniques extends context at fine-tuning or inference time.

### Position Interpolation (Chen et al. 2023)

RoPE frequencies are defined for $[0, L_{train}]$.
To handle $L_{target} > L_{train}$, **interpolate** positions:

$$m' = m \cdot \frac{L_{train}}{L_{target}}$$

This compresses the position range back to $[0, L_{train}]$ without retraining from scratch.
Requires only ~1000 fine-tuning steps to recover quality.

### YaRN Yet another RoPE extensioN (Peng et al. 2023)

YaRN improves on position interpolation with two additions:
1. **Non-uniform interpolation**: different scaling for different RoPE frequency bands
   - Low frequencies (long-range): more interpolation
   - High frequencies (local): keep original (no interpolation)
2. **Attention temperature scaling**: multiply attention logits by $\sqrt{d}/\sqrt{d \cdot t}$
   to compensate for distribution shift

YaRN is used in Mistral 7B v0.2 (32k context) and Mixtral.

### NTK-Aware Interpolation

Rather than scaling positions, change the RoPE base $\theta$:

$$\theta'_j = \left(\theta \cdot \left(\frac{L_{target}}{L_{train}}\right)^{d/(d-2)}\right)^{-2j/d}$$

Higher $\theta$ leads to lower frequencies and less aliasing at long distances.
Can be applied without any fine-tuning (zero-shot extension).

### LongRoPE and LongLoRA

**LongRoPE** (Ding et al. 2024): progressive non-uniform position interpolation with
short and long extension factors. Achieves 2M tokens on LLaMA.

**LongLoRA** (Chen et al. 2023): efficient long-context fine-tuning using:
- **Shifted Sparse Attention (S2-Attn)** during fine-tuning (not inference)
- Standard full attention at inference time
- Only LoRA adapters are trained

### Summary Table

| Method | Fine-tuning needed | Quality | Max context |
|--------|-------------------|---------|-------------|
| Position Interpolation | ~1000 steps | good | 8-32k |
| NTK-aware | none (zero-shot) | moderate | 4-16k |
| YaRN | moderate | very good | 128k |
| LongRoPE | moderate | excellent | 2M |
| LongLoRA | LoRA only | good | 100k+ |
| Ring Attention | full | best | unlimited |

### Practical Tips

- For inference beyond 4x training length, always apply NTK scaling even without fine-tuning.
- For production long-context deployment, combine YaRN fine-tuning with Flash Attention 2.
- Perplexity alone is insufficient test needle-in-a-haystack and retrieval benchmarks.
- Memory grows linearly with context for KV cache; budget accordingly.

## Additional Learning Resources

### Foundational Papers

| Paper | Topic | arXiv |
|-------|-------|-------|
| GQA: Training Generalized Multi-Query Transformer... | Grouped Query Attention | 2305.13245 |
| Fast Transformer Decoding (Shazeer 2019) | Multi-Query Attention | 1911.02150 |
| RoFormer: Enhanced Transformer with Rotary... | RoPE | 2104.09864 |
| Train Short, Test Long: ALiBi | ALiBi | 2108.12409 |
| Mistral 7B | SWA + GQA in practice | 2310.06825 |
| Mixtral of Experts | MoE with top-2 routing | 2401.04088 |
| Switch Transformers | Top-1 MoE routing | 2101.03961 |
| Mamba: Linear-Time Sequence Modeling... | Selective SSM | 2312.00752 |
| Transformers are SSMs: Mamba-2 | SSM-Attention duality | 2405.21060 |
| RWKV: Reinventing RNNs for the Transformer Era | RWKV | 2305.13048 |
| Retentive Network: A Successor to Transformer | RetNet | 2307.08621 |
| FlashAttention: Fast and Memory-Efficient... | Flash Attention | 2205.14135 |
| GLU Variants Improve Transformer (Shazeer) | SwiGLU | 2002.05202 |
| YaRN: Efficient Context Window Extension | YaRN RoPE | 2309.00071 |
| Sparse Upcycling | Dense-to-MoE conversion | 2212.09535 |
| Ring Attention with Blockwise... | Distributed long context | 2310.01889 |
| Extending Context Window via Positional Interp | Position Interpolation | 2306.15595 |
| LongLoRA | Efficient long-context fine-tuning | 2309.12307 |
| Branch-Train-MiX | BTX domain MoE | 2312.11539 |

### Software Libraries

- **HuggingFace Transformers**: source implementations of GQA, RoPE, SwiGLU, RMSNorm
  - `transformers/models/llama/modeling_llama.py` LLaMA reference implementation
- **flash-attn**: `pip install flash-attn` GPU-optimised FlashAttention 1/2/3
- **mamba-ssm**: `pip install mamba-ssm` official Mamba implementation
- **torchtune**: Meta's fine-tuning library with modern arch components
- **xformers**: `pip install xformers` memory-efficient attention, fused ops

### Courses and Tutorials

- **Andrej Karpathy "Let's build GPT from scratch"** (YouTube): foundational attention walkthrough
- **Andrej Karpathy minGPT / nanoGPT**: minimal clean implementations
- **Stanford CS224N**: Natural Language Processing with Deep Learning
- **Hugging Face NLP Course** (huggingface.co/learn/nlp-course): practical Transformers
- **Lilian Weng's blog** (lilianweng.github.io): excellent survey posts on attention and MoE
  - "Attention? Attention!" comprehensive attention overview
  - "Large Transformer Model Inference Optimization"
  - "Mixture of Experts" survey

### Key GitHub Repositories

- `facebookresearch/llama` official LLaMA implementations
- `state-spaces/mamba` Mamba official code
- `BlinkDL/RWKV-LM` RWKV training code
- `microsoft/torchscale` RetNet and other architectures
- `Dao-AILab/flash-attention` FlashAttention